# YieldGuard — Wafer Defect Vision Transformer (Colab T4 GPU)

Trains `WaferViT` (Compact Vision Transformer for 64×64 single-channel wafer maps) on the WM-811K benchmark.

### Why ViT over CNN for Wafer Defect Recognition?
- Standard CNNs rely on local receptive fields, making it difficult to maintain spatial continuity across thin linear defects (e.g. **Scratch** — Case Study 3) or annular boundaries (**Donut** / **Edge-Ring**).
- ViT uses global self-attention across 8×8 patches (64 total tokens), capturing long-range correlations across the full wafer diameter from layer 1.

**Hardware:** Select **Runtime → Change runtime type → T4 GPU** before running. Training takes ~4-6 minutes on T4.

In [ ]:
# 1. Upload colab/wm811k_64.npz (14.5 MB) from the repo
from google.colab import files
print("Upload wm811k_64.npz from src/models/vision/colab/wm811k_64.npz:")
up = files.upload()

In [ ]:
# 2. Environment & Dataset Verification
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time, json
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Execution Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE=="cuda" else "(Warning: GPU recommended)")

d = np.load("wm811k_64.npz")
# Stored as uint8 {0,1,2}; map to float32 in [0, 1]
X_train = d["X_train"].astype(np.float32) / 2.0
X_val   = d["X_val"].astype(np.float32) / 2.0
y_train = d["y_train"].astype(np.int64)
y_val   = d["y_val"].astype(np.int64)

CLASSES = ["Center", "Donut", "Edge-Loc", "Edge-Ring", "Local", "Random", "Scratch", "Near-full", "None"]
print(f"Dataset loaded: Train={X_train.shape}, Val={X_val.shape}")
for i, c in enumerate(CLASSES):
    print(f"  {c:<12}: {int((y_train == i).sum()):>6} train samples")

In [ ]:
# 3. Vision Transformer Architecture (WaferViT)
class WaferViT(nn.Module):
    def __init__(self, img_size=64, patch_size=8, in_chans=1, num_classes=9, embed_dim=192, depth=4, num_heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2  # 64 patches
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size, bias=False)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(p=dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)  # (B, 64, embed_dim)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        x = self.encoder(x)
        x = self.norm(x)
        return self.head(x[:, 0])

model = WaferViT().to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"WaferViT initialized with {params:,} trainable parameters")

In [ ]:
# 4. PyTorch Dataset with Rotational Invariance & Balanced Sampler
class WaferDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.from_numpy(X).unsqueeze(1)
        self.y = torch.from_numpy(y)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        img = self.X[idx]
        if self.augment:
            k = torch.randint(0, 4, (1,)).item()
            img = torch.rot90(img, k, [1, 2])
            if torch.rand(1).item() > 0.5: img = torch.flip(img, [2])
        return img, self.y[idx]

# Balanced class weights for sampler
class_counts = np.bincount(y_train, minlength=9)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[y_train]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(WaferDataset(X_train, y_train, augment=True), batch_size=64, sampler=sampler, num_workers=2)
val_loader   = DataLoader(WaferDataset(X_val, y_val, augment=False), batch_size=128, shuffle=False, num_workers=2)

In [ ]:
# 5. Training Loop with Cosine Annealing & Early Checkpointing
EPOCHS = 35
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_macro_f1 = 0.0
best_model_path = "best_model_vit.pt"

print(f"Beginning training for {EPOCHS} epochs...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(labels)
    scheduler.step()
    train_loss /= len(y_train)

    # Validation
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            out = model(imgs)
            preds.extend(out.argmax(dim=-1).cpu().numpy())
            targets.extend(labels.numpy())

    macro_f1 = f1_score(targets, preds, average="macro")
    dur = time.time() - t0

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        torch.save(model.state_dict(), best_model_path)
        saved_marker = "* (SAVED BEST)"
    else:
        saved_marker = ""

    if epoch % 5 == 0 or epoch == EPOCHS:
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Macro-F1: {macro_f1:.4f} ({dur:.1f}s) {saved_marker}")

In [ ]:
# 6. Final Evaluation Breakdown & Classification Report
model.load_state_dict(torch.load(best_model_path))
model.eval()
preds, targets = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        out = model(imgs.to(DEVICE))
        preds.extend(out.argmax(dim=-1).cpu().numpy())
        targets.extend(labels.numpy())

print("=== BEST WAFERVIT VALIDATION PERFORMANCE ===")
print(classification_report(targets, preds, target_names=CLASSES, digits=4))

# Export holdout metrics json
metrics = {
    "architecture": "WaferViT (ViT-Tiny)",
    "macro_f1": float(f1_score(targets, preds, average='macro')),
    "params": int(params),
    "dataset": "WM-811K (64x64)"
}
with open("holdout_results_vit.json", "w") as f:
    json.dump(metrics, f, indent=2)

In [ ]:
# 7. Download trained weights to place in src/models/vision/checkpoints/
files.download(best_model_path)
files.download("holdout_results_vit.json")
print("Drop best_model_vit.pt into src/models/vision/checkpoints/best_model_vit.pt")